# Notebook 04 — Building a Transformer Model

## Mục tiêu

Xây dựng một Transformer Model từ các thành phần đã học:

1. Token Embedding
2. Positional Encoding
3. Multi-Head Attention
4. Transformer Encoder Blocks
5. Output Layer

Quy trình thực hiện:

Prediction → Implementation → Experiment → Observation → Analysis

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

PyTorch version: 2.14.1+cpu
Device: cpu


## Token Embedding

### Mục tiêu

Chuyển mỗi token ID thành một vector có số chiều cố định thông qua lớp Embedding.

### Kiến thức

Token Embedding là một bảng tham số có thể học. Mỗi token ID tương ứng với một vector biểu diễn.

### Prediction

- Đầu vào là một tensor chứa các token ID.
- Đầu ra là các vector embedding.
- Số lượng token không thay đổi.
- Chiều cuối của tensor đầu ra bằng `d_model`.

In [2]:
vocab_size = 100
d_model = 16

embedding = nn.Embedding(vocab_size, d_model)

token_ids = torch.tensor([
    [1, 5, 8, 2, 9],
    [4, 7, 3, 6, 0]
])

embeddings = embedding(token_ids)

print("Token IDs shape:", token_ids.shape)
print("Embeddings shape:", embeddings.shape)
print("First token embedding:")
print(embeddings[0, 0])

Token IDs shape: torch.Size([2, 5])
Embeddings shape: torch.Size([2, 5, 16])
First token embedding:
tensor([-0.0122, -0.5853,  0.6433, -0.7490,  0.1089,  0.8967, -3.0472, -0.6032,
         1.1199, -0.9666, -0.9998, -0.2428, -0.8881,  1.2822,  2.0106, -1.7020],
       grad_fn=<SelectBackward0>)


In [3]:
vocab_size = 100
d_model = 16

embedding = nn.Embedding(vocab_size, d_model)

token_ids = torch.tensor([
    [1, 5, 8, 2, 9],
    [4, 7, 3, 6, 0]
])

embeddings = embedding(token_ids)

print("Token IDs shape:", token_ids.shape)
print("Embeddings shape:", embeddings.shape)
print("First token embedding:")
print(embeddings[0, 0])

Token IDs shape: torch.Size([2, 5])
Embeddings shape: torch.Size([2, 5, 16])
First token embedding:
tensor([-0.7786,  0.2247,  0.8489,  0.4803,  0.9572, -0.0990, -0.3764,  0.2320,
        -0.9395,  0.2103, -0.5731, -1.5162,  0.6581,  0.4092,  0.4223,  0.1545],
       grad_fn=<SelectBackward0>)


### Nhận xét kết quả (Observation)

- Kích thước Token IDs: (2, 5).
- Kích thước Embeddings: (2, 5, 16).
- Số chiều embedding: 16.

Lớp Token Embedding chuyển mỗi token ID thành một vector biểu diễn 16 chiều.

Số lượng token được giữ nguyên, trong khi mỗi token được mở rộng thành một vector đặc trưng. Các vector embedding là tham số có thể học trong quá trình huấn luyện.

## Combining Token Embedding and Positional Encoding

### Mục tiêu

Kết hợp thông tin về nội dung token và vị trí của token trong sequence.

### Prediction

- Token Embedding cung cấp thông tin về nội dung.
- Positional Encoding cung cấp thông tin về vị trí.
- Tensor sau khi cộng hai thành phần vẫn giữ nguyên kích thước.
- Cùng một token ID ở hai vị trí khác nhau có thể có biểu diễn cuối cùng khác nhau.

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        sequence_length = x.size(1)
        return x + self.pe[:, :sequence_length, :]

In [7]:
positional_encoding = PositionalEncoding(d_model=16)

embeddings_with_position = positional_encoding(embeddings)

print("Embedding shape:", embeddings.shape)
print("Output shape:", embeddings_with_position.shape)

print("\nToken at position 0:")
print(embeddings_with_position[0, 0])

print("\nToken at position 1:")
print(embeddings_with_position[0, 1])

Embedding shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])

Token at position 0:
tensor([-0.7786,  1.2247,  0.8489,  1.4803,  0.9572,  0.9010, -0.3764,  1.2320,
        -0.9395,  1.2103, -0.5731, -0.5162,  0.6581,  1.4092,  0.4223,  1.1545],
       grad_fn=<SelectBackward0>)

Token at position 1:
tensor([ 1.0707,  1.6299, -0.9490,  0.7406,  1.9808,  3.0661,  0.3637,  1.8664,
        -1.0736,  0.5240,  2.5659,  0.9889,  0.8814,  2.6682, -0.4539,  0.6700],
       grad_fn=<SelectBackward0>)


### Nhận xét kết quả (Observation)

- Kích thước Embedding ban đầu: (2, 5, 16).
- Kích thước sau Positional Encoding: (2, 5, 16).

Positional Encoding được cộng trực tiếp vào Token Embedding mà không làm thay đổi kích thước tensor.

Các giá trị biểu diễn thay đổi theo vị trí của token. Nhờ đó, mô hình có thể tiếp nhận cả thông tin nội dung và thông tin thứ tự trong sequence.

Kết quả phù hợp với dự đoán ban đầu.

## Transformer Encoder

### Mục tiêu

Xây dựng Transformer Encoder Block và kiểm tra khả năng xử lý chuỗi đầu vào.

### Cấu trúc

Input
→ Multi-Head Attention
→ Residual Connection + LayerNorm
→ Feed Forward Network
→ Residual Connection + LayerNorm
→ Output

### Prediction

- Encoder giữ nguyên số lượng token.
- Kích thước vector đầu ra bằng d_model.
- Mỗi token có thể trao đổi thông tin với các token khác thông qua Self-Attention.

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, sequence_length, _ = x.shape
        x = x.reshape(batch_size, sequence_length, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (self.d_k ** 0.5)

        weights = F.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)
        output = output.transpose(1, 2)

        batch_size, sequence_length, _, _ = output.shape
        output = output.reshape(batch_size, sequence_length, self.d_model)

        return self.W_o(output), weights

In [9]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        return self.linear2(x)

In [10]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attention_output, weights = self.attention(x)
        x = self.norm1(x + attention_output)

        ffn_output = self.ffn(x)
        output = self.norm2(x + ffn_output)

        return output, weights

### Prediction

- Encoder Block sẽ giữ nguyên kích thước tensor đầu vào.
- Attention Weights có kích thước (batch_size, num_heads, sequence_length, sequence_length).
- FFN xử lý từng token nhưng không thay đổi số lượng token.

In [11]:
d_model = 16
num_heads = 4
d_ff = 64

encoder = TransformerEncoderBlock(d_model, num_heads, d_ff)

encoder_output, attention_weights = encoder(embeddings_with_position)

print("Input shape:", embeddings_with_position.shape)
print("Output shape:", encoder_output.shape)
print("Attention weights shape:", attention_weights.shape)
print("Total parameters:", sum(p.numel() for p in encoder.parameters()))

Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])
Total parameters: 3280


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (2, 5, 16).
- Kích thước đầu ra: (2, 5, 16).
- Attention Weights: (2, 4, 5, 5).
- Tổng số tham số: 3.280.

Transformer Encoder Block giữ nguyên kích thước tensor đầu vào và đầu ra.

Multi-Head Attention cho phép mỗi token thu thập thông tin từ các token khác thông qua 4 attention heads. FFN tiếp tục xử lý đặc trưng của từng token. Residual Connection và Layer Normalization hỗ trợ kết hợp và chuẩn hóa biểu diễn.

Kết quả xác nhận các thành phần đã được kết hợp thành một Encoder Block hoàn chỉnh.

## Building a Transformer Encoder Model

### Mục tiêu

Kết hợp các thành phần đã xây dựng thành một Transformer Encoder Model.

### Kiến trúc

Token IDs
→ Token Embedding
→ Positional Encoding
→ Multiple Encoder Blocks
→ Output Layer

### Prediction

- Đầu ra vẫn giữ số lượng token ban đầu.
- Mỗi Encoder Block tiếp tục cập nhật biểu diễn của sequence.
- Output Layer chuyển vector d_model thành vector có số chiều vocab_size.

In [12]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)

        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        x = self.embedding(token_ids)
        x = self.positional_encoding(x)

        attention_weights = []

        for encoder in self.encoder_blocks:
            x, weights = encoder(x)
            attention_weights.append(weights)

        output = self.output_layer(x)

        return output, attention_weights

In [13]:
vocab_size = 100
d_model = 16
num_heads = 4
d_ff = 64
num_layers = 3

model = TransformerModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

print(model)

TransformerModel(
  (embedding): Embedding(100, 16)
  (positional_encoding): PositionalEncoding()
  (encoder_blocks): ModuleList(
    (0-2): 3 x TransformerEncoderBlock(
      (attention): MultiHeadAttention(
        (W_q): Linear(in_features=16, out_features=16, bias=True)
        (W_k): Linear(in_features=16, out_features=16, bias=True)
        (W_v): Linear(in_features=16, out_features=16, bias=True)
        (W_o): Linear(in_features=16, out_features=16, bias=True)
      )
      (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): FeedForwardNetwork(
        (linear1): Linear(in_features=16, out_features=64, bias=True)
        (relu): ReLU()
        (linear2): Linear(in_features=64, out_features=16, bias=True)
      )
      (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (output_layer): Linear(in_features=16, out_features=100, bias=True)
)


In [14]:
token_ids = torch.tensor([
    [1, 5, 8, 2, 9],
    [4, 7, 3, 6, 0]
])

output, attention_weights = model(token_ids)

print("Input shape:", token_ids.shape)
print("Output shape:", output.shape)
print("Number of encoder blocks:", len(model.encoder_blocks))
print("Number of attention weight tensors:", len(attention_weights))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Input shape: torch.Size([2, 5])
Output shape: torch.Size([2, 5, 100])
Number of encoder blocks: 3
Number of attention weight tensors: 3
Total parameters: 13140


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (2, 5).
- Kích thước đầu ra: (2, 5, 100).
- Số Encoder Blocks: 3.
- Số Attention Weight tensors: 3.
- Tổng số tham số: 13.140.

Transformer Model đã kết hợp thành công Token Embedding, Positional Encoding, các Encoder Blocks và Output Layer.

Mô hình giữ nguyên số lượng token trong sequence. Output Layer chuyển biểu diễn 16 chiều của mỗi token thành vector 100 chiều, tương ứng với kích thước từ vựng.

Kết quả xác nhận mô hình có thể xử lý đầu vào và tạo ra điểm số dự đoán cho từng token tại mỗi vị trí.

## Prediction Probabilities

### Mục tiêu

Chuyển các giá trị đầu ra (Logits) thành xác suất bằng Softmax.

### Prediction

- Tổng xác suất tại mỗi vị trí token bằng 1.
- Mỗi giá trị xác suất nằm trong khoảng từ 0 đến 1.
- Token có xác suất cao nhất là token được mô hình ưu tiên dự đoán.

In [15]:
probabilities = F.softmax(output, dim=-1)

predicted_tokens = torch.argmax(probabilities, dim=-1)

print("Probabilities shape:", probabilities.shape)
print("Sum of probabilities:")
print(probabilities.sum(dim=-1))

print("\nPredicted token IDs:")
print(predicted_tokens)

Probabilities shape: torch.Size([2, 5, 100])
Sum of probabilities:
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)

Predicted token IDs:
tensor([[75, 68, 22, 75, 75],
        [75, 16, 31, 31, 11]])


### Nhận xét kết quả (Observation)

- Kích thước Probabilities: (2, 5, 100).
- Tổng xác suất tại mỗi vị trí token bằng 1.
- Mỗi vị trí có 100 giá trị xác suất.
- Argmax trả về một token ID có xác suất cao nhất tại mỗi vị trí.

Softmax chuyển Logits thành phân phối xác suất trên toàn bộ vocabulary. Argmax lựa chọn token có xác suất cao nhất.

Kết quả xác nhận mô hình có thể tạo ra phân phối xác suất và dự đoán token tại từng vị trí trong sequence.

Do mô hình chưa được huấn luyện, các dự đoán hiện tại chưa phản ánh khả năng hiểu ngôn ngữ.

## Loss and Backpropagation

### Mục tiêu

Kiểm tra khả năng tính Loss và lan truyền gradient qua toàn bộ Transformer Model.

### Prediction

- Cross Entropy Loss đo sự khác biệt giữa dự đoán và nhãn thực tế.
- Backpropagation truyền gradient về các tham số có thể học.
- Embedding, Attention, FFN và Output Layer đều có thể nhận gradient.

In [16]:
targets = torch.randint(0, vocab_size, (2, 5))

loss = F.cross_entropy(
    output.reshape(-1, vocab_size),
    targets.reshape(-1)
)

model.zero_grad()
loss.backward()

print("Targets:")
print(targets)

print("\nLoss:", loss.item())

print("\nEmbedding gradient:",
      model.embedding.weight.grad.norm().item())

print("Attention gradient:",
      model.encoder_blocks[0].attention.W_q.weight.grad.norm().item())

print("FFN gradient:",
      model.encoder_blocks[0].ffn.linear1.weight.grad.norm().item())

print("Output layer gradient:",
      model.output_layer.weight.grad.norm().item())

Targets:
tensor([[45, 53,  0, 22, 20],
        [27, 67, 83, 16, 40]])

Loss: 4.702550888061523

Embedding gradient: 0.1784006506204605
Attention gradient: 0.08943408727645874
FFN gradient: 0.2421407252550125
Output layer gradient: 1.2501211166381836


### Nhận xét kết quả (Observation)

- Cross Entropy Loss: 4.7026.
- Embedding Gradient Norm: 0.1784.
- Attention Gradient Norm: 0.0894.
- FFN Gradient Norm: 0.2421.
- Output Layer Gradient Norm: 1.2501.

Hàm Cross Entropy Loss tính toán được sai số giữa Logits và các nhãn mục tiêu.

Các thành phần Embedding, Attention, FFN và Output Layer đều nhận được gradient khác 0. Điều này xác nhận quá trình Backpropagation có thể truyền tín hiệu học qua toàn bộ mô hình.

Loss hiện tại gần mức tham chiếu ln(100) vì mô hình mới khởi tạo và chưa trải qua quá trình huấn luyện.

## One Training Step

### Mục tiêu

Thực hiện một bước cập nhật tham số bằng Adam Optimizer.

### Prediction

Sau khi cập nhật tham số, đầu ra của mô hình có thể thay đổi vì các trọng số đã được điều chỉnh dựa trên gradient.

In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()

optimizer.zero_grad()

output, _ = model(token_ids)

loss = F.cross_entropy(
    output.reshape(-1, vocab_size),
    targets.reshape(-1)
)

loss_before = loss.item()

loss.backward()
optimizer.step()

new_output, _ = model(token_ids)

new_loss = F.cross_entropy(
    new_output.reshape(-1, vocab_size),
    targets.reshape(-1)
)

print("Loss before update:", loss_before)
print("Loss after update:", new_loss.item())

Loss before update: 4.702550888061523
Loss after update: 4.611601829528809


### Nhận xét kết quả (Observation)

- Loss trước cập nhật: 4.70255.
- Loss sau cập nhật: 4.61160.
- Mức giảm Loss: 0.09095.

Sau một bước cập nhật bằng Adam Optimizer, Loss giảm khoảng 1.93%.

Kết quả cho thấy gradient được tính toán và sử dụng để điều chỉnh các tham số của Transformer Model. Mô hình tạo ra dự đoán tốt hơn trên cùng batch dữ liệu sau khi cập nhật.

Thí nghiệm xác nhận quy trình cơ bản của quá trình huấn luyện gồm Forward Propagation, Loss Calculation, Backpropagation và Parameter Update hoạt động đúng.